一、最常用基础语法

1.列表推导式（像写集合一样生成数字）

1.1 数学里的 $\{x^2 \mid x \in\{0,1,\dots,9\}\text{且}x\text{是偶数}\}$，在python里面写作：

In [1]:
evens_squared=[x**2 for x in range(10) if x % 2 == 0]
print(evens_squared)

[0, 4, 16, 36, 64]


2.函数与数学映射（def与lambda）

2.1 写一个函数就是定义一个映射f(x):

In [2]:
#方法一
def f(x):
    return x**3-2*x-2
#方法二
g=lambda x:x**2+1

3.数值差分（用数学逻辑写出你的第一个数值工具）

3.1 根据导数定义$f'(x)\approx\frac{f(x+h)-f(x-h)}{2h}$,写一个通用数值导数生成器（返回一个函数）

In [3]:
def derivative(f,h=1e-5):                    #f是占位符，代表将来要进去的函数。h是默认步长10^(-5)
    return lambda x:(f(x+h)-f(x-h))/(2*h)
df = derivative(lambda x:x**2)
print("数值导数结果:",df(3.0))                #在x=3处求导

数值导数结果: 6.000000000039306


二、广播机制

1.给定两个点集$X\in \mathbb{R}^{M\times D}$和$Y \in \mathbb{R}^{N\times D}$，计算他们之间的欧几里得距离矩阵$D \in \mathbb{R}^{M\times N}$,其中$D_{ij}=\|X_i-Y_j\|_2$。

解法1：广播升维法：利用切片增加维度$X_{[:,None,:]}$（形状变为$M\times 1\times D$）与$Y_{[None,:,:]}$（形状变为$1\times N\times D$),相减后沿最后一维求范数：
$$
D_{ij}=\sqrt{\sum_{k=1}^{D}(X_{ik}-Y_{jk})^2}
$$
解法2：矩阵分解展开法：利用恒等式$\|x-y\|^2=\|X\|^2+\|Y\|^2-2x^\top y$。直接通过点积$X@Y.T$结合每行的二范数平方向量进行广播加法（在大样本下加速更快）

In [8]:
#解法1：
import numpy as np
X=np.array([[0,0],
           [3,4]])
Y=np.array([[1,1],
            [2,2]])
diff=X[:,None,:]-Y[None,:,:]      #切片
print("所有点对的坐标差:\n",diff)  #换行符
squared=diff**2
summed=np.sum(squared,axis=-1)    #最后一维
dist=np.sqrt(summed)
print("最终的距离:\n",dist)

所有点对的坐标差:
 [[[-1 -1]
  [-2 -2]]

 [[ 2  3]
  [ 1  2]]]
最终的距离:
 [[1.41421356 2.82842712]
 [3.60555128 2.23606798]]


2. 切片及高级切片

In [6]:
import numpy as np
A=np.array([[0,1,2,3],
            [4,5,6,7],
            [8,9,10,11],
            [12,13,14,15]])
B=A[1:3,1:3]    #左闭右开 #切片格式记住
print(B)

[[ 5  6]
 [ 9 10]]


三、@运算符与矩阵相乘

1.数值分析经典算法————幂法(求主特征值与特征向量)

目标：实现一个向量化的幂迭代算法，求解任意对称正定实矩阵 $A \in \mathbb{R}^{n \times n}$ 的谱半径（绝对值最大的特征值 $\lambda_{\max}$）及对应特征向量。

数学步骤：

初始化随机非零列向量 $v_0 \in \mathbb{R}^n$ 且 $\Vert{}v_0\Vert{}_2 = 1$。

迭代直到残差满足条件：$$w_{k+1} = A v_k, \quad v_{k+1} = \frac{w_{k+1}}{\Vert{}w_{k+1}\Vert{}_2}$$计算瑞利商（Rayleigh Quotient）估计特征值：$$\lambda_k = \frac{v_k^\top A v_k}{v_k^\top v_k} = v_k^\top A v_k$$

范数计算用 np.linalg.norm(..., ord=2)

In [2]:
import numpy as np
A=np.array([[4,1,1],
            [1,3,0],
            [1,0,2]])
v=np.ones(3)
v=v/np.linalg.norm(v)

for _ in range(20):
    w=A@v                 #做矩阵乘法
    v=w/np.linalg.norm(w) #求向量二范数 
lambda_max=v@A@v
print(lambda_max)

4.87938524157158


四、数组按轴规约与keepdims机制

1.计算每个样本向量的二范数平方

假设 $X$ 是一个包含 3 个点、每个点有 2 个坐标的矩阵,现在要计算每个点的坐标平方和（即每一行的平方和）：

In [12]:
import numpy as np
X=np.array([[3,4],
            [1,2],
            [0,5]])
A=np.sum(X**2,axis=1,keepdims=True)  #axis=0 表示沿着行(竖向)压缩，axis=1 表示沿着列(横向)压缩。
B=np.sum(X**2,axis=1)
print(A,B)

[[25]
 [ 5]
 [25]] [25  5 25]


五、布尔掩码与条件索引

1、截断过小的数值误差（小值置零）

In [20]:
import numpy as np
R=np.array([[2.5,-1e-8],
            [3e-7,-4.2]])
eps=1e-5
mask=np.abs(R)<eps   #这个输出的是布尔类型

R[mask]=0.0
print(mask,"\n",R)

[[False  True]
 [ True False]] 
 [[ 2.5  0. ]
 [ 0.  -4.2]]


六、对角矩阵构造与上/下三角提取

在计算数学里，解线性方程组的迭代法（如 Jacobi、Gauss-Seidel 迭代）核心

就是把矩阵 $A$ 拆解为 $A = D - L - U$（对角矩阵、严格下三角、严格上三角）。

1.矩阵的经典分裂（Jacobi 迭代的前置步骤）

In [2]:
import numpy as np
A=np.array([[10,-1,2],
            [-1,11,-1],
            [2,-1,10]])
diag_A=np.diag(A)
D=np.diag(diag_A)    #提取对角元素并构造对角矩阵 $D$
                    #np.diag 是一个具备“双向功能”的神奇函数：

                    #传给它一个 2D 矩阵，它帮你提取出对角线上的元素，返回 1D 向量。

                    #传给它一个 1D 向量，它帮你构造出一个对角矩阵，非对角线位置全部填 0。

L = -np.tril(A, k=-1) #NumPy 提供了提取三角矩阵的内置函数：np.tril（下三角，lower）和 np.triu（上三角，upper）。
                       #参数 k 表示偏离主对角线的偏移量（$k=0$ 为主对角线）。
U = -np.triu(A, k=1)
print(D,"\n",L,"\n",U)
print(np.allclose(A,D-L-U))

[[10  0  0]
 [ 0 11  0]
 [ 0  0 10]] 
 [[ 0  0  0]
 [ 1  0  0]
 [-2  1  0]] 
 [[ 0  1 -2]
 [ 0  0  1]
 [ 0  0  0]]
True


七、向量化解对角方程组与求逆陷阱

在数值分析和计算数学中，有一条铁律：
除非理论推导需要，否则在代码里永远不要直接显式求逆矩阵（Avoid np.linalg.inv）

1.对角矩阵方程的高效求解

In [1]:
import numpy as np

diag_d = np.array([10.0, 11.0, 10.0])
D = np.diag(diag_d)
r = np.array([5.0, 22.0, 30.0])       # 右端项残差向量 r，即为右端向量
x = r / diag_d
print(x)

[0.5 2.  3. ]


2.一般稠密线性方程组 $A x = b$ 的标准做法：(没有大量 0，也没有特殊的对角结构)

In [2]:
import numpy as np
A=np.array([[2.0,1.0,1.0],
            [4.0,-6.0,0.0],
            [-2.0,7.0,2.0]])
b=np.array([5.0,-2.0,9.0])
x=np.linalg.solve(A,b)      #底层自动完成 LU 分解 + 前代 + 回代
residual=b-A@x
print("解x:",x)
print("残差向量",residual)
print("残差二范数:",np.linalg.norm(residual))

解x: [1. 1. 2.]
残差向量 [0. 0. 0.]
残差二范数: 0.0


3.如果你有一个固定的系统矩阵 $A$，但需要对成千上万个不同的 $b$ 重复求解

In [ ]:
import numpy as np
from scipy.linalg import lu_factor, lu_solve
A=np.array([[2.0,1.0,1.0],            
            [4.0,-6.0,0.0],            
            [-2.0,7.0,2.0]])
b=np.array([5.0,-2.0,9.0])
# 1. 一次性分解 (耗时 O(n^3))
lu, piv = lu_factor(A)     #把矩阵 $A$ 拆解为 $PA = LU$

# 2. 后续在循环中面对不同的 b_i 时，极速求解 (单次仅耗时 O(n^2))
x1 = lu_solve((lu, piv), b1)
x2 = lu_solve((lu, piv), b2)

In [3]:
import numpy as np
from scipy.linalg import lu_factor, lu_solve
import time

# 构造一个 1000x1000 的大矩阵（模拟工程系统）
n = 1000
A = np.random.randn(n, n) + np.eye(n) * 10  # 对角占优，保证可逆且稳定

# 1. 循环外：一次性 LU 分解（只需做一次 O(n^3) 运算）
lu, piv = lu_factor(A)

# 2. 循环内：模拟 100 个时间步的求解
# 每次只需要做 O(n^2) 的前代和回代，速度极快
solutions = []
start_time = time.time()

for t in range(100):
    b_t = np.random.randn(n)           # 当前时刻的外部输入
    x_t = lu_solve((lu, piv), b_t)     # 🚀 毫秒级极速回代
    solutions.append(x_t)

print(f"100 次求解总耗时: {time.time() - start_time:.4f} 秒")

100 次求解总耗时: 0.0336 秒


八、三对角矩阵的向量化生成与离散拉普拉斯算子

1.假设我们要构造一个 $5 \times 5$ 的离散拉普拉斯矩阵 $A$：主对角线（$k=0$）：全为 -2.0（对应中心点权重 $-2$）上副对角线（$k=1$）：全为 1.0（对应右侧邻点权重 $+1$）下副对角线（$k=-1$）：全为 1.0（对应左侧邻点权重 $+1$）$$A = \begin{bmatrix} -2 &  1 &  0 &  0 &  0 \\  1 & -2 &  1 &  0 &  0 \\  0 &  1 & -2 &  1 &  0 \\  0 &  0 &  1 & -2 &  1 \\  0 &  0 &  0 &  1 & -2 \end{bmatrix}$$

In [5]:
A = -2 * np.eye(5) + np.diag(np.ones(4), k=1) + np.diag(np.ones(4), k=-1)
print(A)

[[-2.  1.  0.  0.  0.]
 [ 1. -2.  1.  0.  0.]
 [ 0.  1. -2.  1.  0.]
 [ 0.  0.  1. -2.  1.]
 [ 0.  0.  0.  1. -2.]]


九、二维网格构建与向量化函数评估

1.题目：计算二维势场（Potential Field）假设要在二维空间区域 $[0, 1] \times [0, 1]$ 上求解一个静电势函数：$$\Phi(x, y) = \sin(\pi x) \cdot \cos(\pi y)$$

In [6]:
import numpy as np

x = np.linspace(0, 1, 4)  # array([0.        , 0.33333333, 0.66666667, 1.        ])
y = np.linspace(0, 1, 4)  # 同上
X, Y = np.meshgrid(x, y)
Phi = np.sin(np.pi * X) * np.cos(np.pi * Y)
print(Phi)

[[ 0.00000000e+00  8.66025404e-01  8.66025404e-01  1.22464680e-16]
 [ 0.00000000e+00  4.33012702e-01  4.33012702e-01  6.12323400e-17]
 [-0.00000000e+00 -4.33012702e-01 -4.33012702e-01 -6.12323400e-17]
 [-0.00000000e+00 -8.66025404e-01 -8.66025404e-01 -1.22464680e-16]]


在 NumPy 中，对应位置两个数相乘（Hadamard 积 / 逐元素乘法）使用的是普通算术符号 *

@ 是线性代数中的矩阵内积：行向量乘以列向量再累加求和（$C_{ij} = \sum_k A_{ik} B_{kj}$）

2.进阶内存优化小技巧：indexing='ij' 与 sparse=True在求解大型 PDE 网格时（如 $1000 \times 1000$ 网格）：默认的 meshgrid 会在内存里完整分配上百万个冗余浮点数。使用 sparse=True 可以生成稀疏广播网格，避免复制内存：